In [1]:
!pip install --upgrade --no-cache-dir transformers
!pip install -q accelerate peft bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 106.2 MB/s eta 0:00:00


In [2]:
!pip install -U bitsandbytes
!pip install -U transformers

In [3]:
import transformers
print(transformers.__version__)  # Skal være 4.40.0 eller høyere

4.51.3


In [4]:
!nvidia-smi

Mon May 19 15:41:37 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   48C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
!accelerate config default

accelerate configuration saved at /root/.cache/huggingface/accelerate/default_config.yaml


In [6]:
# Åpne og skriv ut de første 5 linjene
with open("asl_dataset_1x.jsonl", "r", encoding="utf-8") as f:
    for _ in range(5):
        print(f.readline())

{"prompt": "Translate the following to American Sign Language (ASL) structure:\n I’ve never heard of that!", "completion": "NEVER EAR THAT!"}

{"prompt": "Translate the following to American Sign Language (ASL) structure:\n“Now morning” means this morning", "completion": "NOW-MORNING MEANING T-H-I-S M-O-R-N-I-N-G"}

{"prompt": "Translate the following to American Sign Language (ASL) structure:\n(Refers to) Nail clippers", "completion": "NAIL CLIPPERS"}

{"prompt": "Translate the following to American Sign Language (ASL) structure:\nA list of five things", "completion": "LIST-[5]-tap-middle"}

{"prompt": "Translate the following to American Sign Language (ASL) structure:\nA list of four things", "completion": "LIST-[4]-tap-ring"}



In [7]:
import os
from huggingface_hub import HfFolder, login

# Sjekk om token allerede er lagret
if HfFolder.get_token() is None:
    login()
else:
    print("Allerede logget inn på Hugging Face 🤗")


In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch  # Du trenger dette for torch.float16

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# Konfigurasjon for 4-bit kvantisering
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# Last inn tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Last inn modellen
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    trust_remote_code=True,
    quantization_config=quant_config
)


tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [10]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # spesifikke for Mistral
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

print("LoRA-konfigurasjon er lagt til modellen.")

LoRA-konfigurasjon er lagt til modellen.


In [11]:
import pandas as pd
from datasets import Dataset

df = pd.read_json("/content/asl_dataset_1x.jsonl", lines=True)
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.1)

print("Eksempel:", dataset["train"][0])


Eksempel: {'prompt': 'Translate the following to American Sign Language (ASL) structure:\nWhen class is over, do you go back home?', 'completion': 'CLASS FINISH, BACK HOME YOU?'}


In [12]:
def format_and_tokenize(example):
    messages = [
        {"role": "user", "content": example["prompt"]},
        {"role": "assistant", "content": example["completion"]}
    ]
    full_text = tokenizer.apply_chat_template(messages, tokenize=False)

    tokenized = tokenizer(
        full_text,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    return {
        "input_ids": tokenized["input_ids"][0],
        "labels": tokenized["input_ids"][0]  # Vi bruker samme tokens som både input og target
    }

tokenized_dataset = dataset.map(format_and_tokenize)

print("Tokenisering med full chat-template ferdig. Eksempel:")
print(tokenized_dataset["train"][0])



Map:   0%|          | 0/1109 [00:00<?, ? examples/s]

Map:   0%|          | 0/124 [00:00<?, ? examples/s]

Tokenisering med full chat-template ferdig. Eksempel:
{'prompt': 'Translate the following to American Sign Language (ASL) structure:\nWhen class is over, do you go back home?', 'completion': 'CLASS FINISH, BACK HOME YOU?', 'input_ids': [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 733, 16289, 28793, 4335, 10020, 272, 2296, 298, 2556, 9315, 15589, 325, 2109, 28758, 28731, 4693, 28747, 13, 7477, 875, 349, 754, 28725, 511, 368, 576, 852, 1611, 28804, 733, 28748, 16289, 28793, 12296, 4816, 401, 775, 26571, 28725, 365, 4614, 12203, 1574, 15479, 28804, 2], 'labels': [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 

In [13]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./mistral-asl-instruct",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    report_to="none"
    # 🟠 evaluation_strategy fjernet midlertidig
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()

# Lagre LORA-vektene etter trening
trainer.model.save_pretrained("./mistral-asl-instruct")  # Navn på mappen du vil lagre til
tokenizer.save_pretrained("./mistral-asl-instruct")

print("Modellen og tokenizer er lagret.")

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
10,3.565600
20,1.404800
30,1.038800
40,0.816500
50,0.870800
60,0.844400
70,0.753600
80,0.766500
90,0.722800
100,0.728600


Modellen og tokenizer er lagret.


In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, PeftConfig
import torch

# Konfig for QLoRA (4-bit)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# Last PEFT-konfig for å vite hvilken base-modell som ble brukt
peft_config = PeftConfig.from_pretrained("./mistral-asl-instruct")

# Last base-modellen i 4-bit
base_model = AutoModelForCausalLM.from_pretrained(
    peft_config.base_model_name_or_path,
    device_map="auto",
    quantization_config=quant_config,
    trust_remote_code=True
)

# Legg på LoRA-vektene
model = PeftModel.from_pretrained(base_model, "./mistral-asl-instruct")

# Last tokenizer
tokenizer = AutoTokenizer.from_pretrained("./mistral-asl-instruct", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [21]:
messages = [
    {"role": "user", "content": "Translate the following to American Sign Language (ASL) structure:\nWhat is your name?"}
]

# Bygg prompt i [INST] ... [/INST]-stil (Mistral sin stil)
prompt = tokenizer.apply_chat_template(messages, return_tensors="pt").to(model.device)

# Generer svar
output = model.generate(
    prompt,
    max_new_tokens=64,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
)

# Dekod og skriv ut
decoded = tokenizer.decode(output[0], skip_special_tokens=True)
print(decoded)


[INST] Translate the following to American Sign Language (ASL) structure:
What is your name? [/INST] NAME-[what]? (what-name) [question]?


In [18]:
messages = [
    {"role": "user", "content": "Translate the following to American Sign Language (ASL) structure:\nI will go on a ski trip this weekend."}
]

# Bygg prompt i [INST] ... [/INST]-stil (Mistral sin stil)
prompt = tokenizer.apply_chat_template(messages, return_tensors="pt").to(model.device)

# Generer svar
output = model.generate(
    prompt,
    max_new_tokens=64,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
)

# Dekod og skriv ut
decoded = tokenizer.decode(output[0], skip_special_tokens=True)
print(decoded)


[INST] Translate the following to American Sign Language (ASL) structure:
I will go on a ski trip this weekend. [/INST] WEEKEND SKI TRIP GO I WILL. (I'm going on a ski trip this weekend.)


In [22]:
import shutil
from google.colab import files

# Lag zip-fil fra mappen
shutil.make_archive("mistral-asl-instruct", "zip", "./mistral-asl-instruct")

# Last ned zip-filen til din PC
files.download("mistral-asl-instruct.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, PeftConfig
import torch

# Last PEFT-konfig
peft_config = PeftConfig.from_pretrained("./mistral-asl-instruct")

# 4-bit kvantisering
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# Last basemodell
base_model = AutoModelForCausalLM.from_pretrained(
    peft_config.base_model_name_or_path,
    device_map="auto",
    trust_remote_code=True,
    quantization_config=quant_config
)

# Legg til LoRA
model = PeftModel.from_pretrained(base_model, "./mistral-asl-instruct")

# Last tokenizer
tokenizer = AutoTokenizer.from_pretrained("./mistral-asl-instruct", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Prompt-test
messages = [
    {"role": "user", "content": "Translate the following to American Sign Language (ASL) structure:\nI am very hungry and tired today."}
]
prompt = tokenizer.apply_chat_template(messages, return_tensors="pt").to(model.device)
output = model.generate(prompt, max_new_tokens=64)
print(tokenizer.decode(output[0], skip_special_tokens=True))


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[INST] Translate the following to American Sign Language (ASL) structure:
I am very hungry and tired today. [/INST] TODAY I HUNGRY, TIRED, VERY. [Today I'm very hungry and tired.]
